# Retrieval với HUIM

Utility-weighted ItemKNN + global 2-HUI / 3-HUI mining (cross-user).

**Output**: `model/hybrid_huim_candidates.pkl` — top-300 candidates / user, dùng cho `ranking_huim.ipynb`.

## 1. Import & Load Data

In [1]:
import json, pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from itertools import combinations
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Load + dedup + filter (giong retrieval.ipynb goc)
review_file = '../data/Luxury_Beauty_5.json'
reviews = []
for line in tqdm(open(review_file, 'r', encoding='utf-8'), desc="Loading"):
    r = json.loads(line.strip())
    reviews.append({'reviewerID': r.get('reviewerID'), 'asin': r.get('asin'),
                    'overall': r.get('overall'), 'unixReviewTime': r.get('unixReviewTime'),
                    'reviewText': r.get('reviewText',''), 'summary': r.get('summary','')})
df = pd.DataFrame(reviews)
print(f"Raw: {df.shape}")
df = df.drop_duplicates(subset=['reviewerID','asin','unixReviewTime'], keep='first').reset_index(drop=True)
print(f"After dedup: {df.shape}")

min_user, min_item = 5, 3
uc = df['reviewerID'].value_counts(); df = df[df['reviewerID'].isin(uc[uc>=min_user].index)]
ic = df['asin'].value_counts(); df = df[df['asin'].isin(ic[ic>=min_item].index)].reset_index(drop=True)
print(f"After filter: {df.shape} | Users={df['reviewerID'].nunique()} Items={df['asin'].nunique()}")

Loading: 34278it [00:00, 122984.52it/s]


Raw: (34278, 6)
After dedup: (29059, 6)
After filter: (25206, 6) | Users=2782 Items=1399


## 2. Train/Test Split

In [3]:
df = df.sort_values(['reviewerID','unixReviewTime','asin'])
train_rows, ground_truth = [], {}
for user, g in df.groupby('reviewerID'):
    items = g['asin'].tolist()
    if len(items) <= 7:
        ground_truth[user] = items[-1:]; train_rows.append(g.iloc[:-1])
    else:
        ground_truth[user] = items[-2:]; train_rows.append(g.iloc[:-2])
train_df = pd.concat(train_rows).reset_index(drop=True)
print(f"Train: {len(train_df)} | GT users: {len(ground_truth)}")

Train: 21076 | GT users: 2782


## 3. Popularity

In [4]:
popularity = (train_df.groupby('asin')
               .agg(avg_rating=('overall','mean'), cnt=('overall','count')).reset_index())
popularity['score'] = popularity['avg_rating'] * popularity['cnt']
pop_score = dict(zip(popularity['asin'], popularity['score']))
pop_max = popularity['score'].max()

## 4. Content-Based (TF-IDF + SVD)

In [5]:
train_df['review_text'] = (train_df['summary'].fillna('') + '. ' + train_df['reviewText'].fillna('')).str.lower().str.strip()
tfidf = TfidfVectorizer(max_features=30000, ngram_range=(1,2), stop_words='english')
X = tfidf.fit_transform(train_df['review_text'])
svd = TruncatedSVD(n_components=100, random_state=42)
X_svd = svd.fit_transform(X)

item_vectors = {}
for asin, vec in zip(train_df['asin'], X_svd):
    item_vectors.setdefault(asin, []).append(vec)
item_vectors = {k: np.mean(v, axis=0) for k, v in item_vectors.items()}
print(f"Item vectors: {len(item_vectors)}")

Item vectors: 1396


## 5. CF — UTILITY-weighted ItemKNN (HUIM principle)

Cosine cua ma tran rating thay vi binary 0/1.

In [6]:
train_df['interaction'] = (train_df['overall'] >= 4).astype(int)
ui = train_df.pivot_table(index='reviewerID', columns='asin', values='interaction', fill_value=0)
ui_util = train_df[train_df['overall']>=4].pivot_table(
    index='reviewerID', columns='asin', values='overall',
    aggfunc='max', fill_value=0).reindex(columns=ui.columns, index=ui.index, fill_value=0)
item_sim_util = pd.DataFrame(cosine_similarity(ui_util.T),
                             index=ui_util.columns, columns=ui_util.columns)
print(f"UTIL ItemKNN: {item_sim_util.shape}")

UTIL ItemKNN: (1396, 1396)


## 6. HUIM Mining — Global 2-HUI & 3-HUI

https://github.com/RominaShin/High-Utility-Itemset-Mining

https://link.springer.com/chapter/10.1007/11430919_79

https://www.philippe-fournier-viger.com/spmf/Two-Phase.php

In [ ]:
user_items_pos = {}
for _, row in train_df.iterrows():
    if row['overall'] >= 4:
        u, i, r = row['reviewerID'], row['asin'], float(row['overall'])
        d = user_items_pos.setdefault(u, {})
        if r > d.get(i, 0): d[i] = r

# Mine 2-HUIs
pair_util = defaultdict(float); pair_count = defaultdict(int)
for u, items_dict in tqdm(user_items_pos.items(), desc="2-HUI"):
    its = sorted(items_dict.keys())
    for a in range(len(its)):
        ia, ra = its[a], items_dict[its[a]] 
        for b in range(a+1, len(its)):
            ib, rb = its[b], items_dict[its[b]]
            pair_util[(ia,ib)] += ra+rb; pair_count[(ia,ib)] += 1
pair_util = {k:v for k,v in pair_util.items() if pair_count[k] >= 2}
print(f"2-HUIs: {len(pair_util):,}")

# Mine 3-HUIs (Apriori-prune)
triple_util = defaultdict(float); triple_count = defaultdict(int)
for u, items_dict in tqdm(user_items_pos.items(), desc="3-HUI"):
    its = sorted(items_dict.keys())
    for a, b, c in combinations(range(len(its)), 3):
        ia, ib, ic_ = its[a], its[b], its[c]
        if (ia,ib) not in pair_util or (ia,ic_) not in pair_util or (ib,ic_) not in pair_util: continue
        triple_util[(ia,ib,ic_)] += items_dict[ia]+items_dict[ib]+items_dict[ic_]
        triple_count[(ia,ib,ic_)] += 1
triple_util = {k:v for k,v in triple_util.items() if triple_count[k] >= 2}
print(f"3-HUIs: {len(triple_util):,}")

# Build lookups
two_neighbors = defaultdict(dict)
for (a,b), u in pair_util.items():
    two_neighbors[a][b] = u; two_neighbors[b][a] = u
three_partners = defaultdict(list)
for (a,b,c), u in triple_util.items():
    three_partners[a].append((frozenset((b,c)), u))
    three_partners[b].append((frozenset((a,c)), u))
    three_partners[c].append((frozenset((a,b)), u))

2-HUI: 100%|██████████| 2672/2672 [00:00<00:00, 29036.55it/s]


2-HUIs: 11,216


3-HUI: 100%|██████████| 2672/2672 [00:00<00:00, 7821.99it/s] 


3-HUIs: 17,425


## 7. Hybrid Ensemble

In [8]:
CB_W, CF_W, POP_W = 0.25, 0.65, 0.10
HUIM_W3 = 0.10
TOPK = 300

items_master = item_sim_util.columns.tolist()
zero_vec = np.zeros(next(iter(item_vectors.values())).shape[0])
IV = np.stack([item_vectors.get(it, zero_vec) for it in items_master])
iv_idx = {k: i for i, k in enumerate(items_master)}

def base_components(user_id):
    items = items_master
    if user_id not in ui.index:
        return [], items, np.array([pop_score.get(it, 0)/pop_max for it in items])
    seen = ui.loc[user_id]
    seen = seen[seen > 0].index.tolist()
    if not seen:
        return [], items, np.array([pop_score.get(it, 0)/pop_max for it in items])
    cf = item_sim_util[seen].mean(axis=1)
    cf_arr = (cf / cf.max()).values if cf.max() > 0 else cf.values
    seen_idx = np.array([iv_idx[s] for s in seen])
    cb_arr = IV @ IV[seen_idx].sum(axis=0)
    cb_arr = cb_arr / cb_arr.max() if cb_arr.max() > 0 else cb_arr
    pop_arr = np.array([pop_score.get(it, 0)/pop_max for it in items])
    return seen, items, CF_W*cf_arr + CB_W*cb_arr + POP_W*pop_arr

def three_hui_score(seen, items):
    seen_set = set(seen)
    s = np.zeros(len(items))
    for i, it in enumerate(items):
        ps_list = three_partners.get(it)
        if not ps_list: continue
        tot = 0.0
        for ps_, u in ps_list:
            if ps_.issubset(seen_set): tot += u
        s[i] = tot
    m = s.max(); return s/m if m > 0 else s

## 8. Generate Top-300 Candidates

In [9]:
users_all = sorted(set(ui.index.tolist()) | set(ground_truth.keys()))
print(f"Users: {len(users_all)}")

hybrid_huim_output = {}
for u in tqdm(users_all, desc="Candidates"):
    seen, items, base = base_components(u)
    h3 = three_hui_score(seen, items) if seen else np.zeros(len(items))
    scores = base + HUIM_W3 * h3
    order = np.argsort(-scores)[:TOPK]
    hybrid_huim_output[u] = [(items[i], float(scores[i])) for i in order]

Users: 2782


Candidates: 100%|██████████| 2782/2782 [00:41<00:00, 67.77it/s]


## 9. Evaluate Recall@K

In [10]:
def recall_at_k(preds, gt, k):
    rs = []
    for u, gti in gt.items():
        if u not in preds: continue
        p = [i for i, _ in preds[u][:k]]
        rs.append(len(set(p) & set(gti)) / len(gti))
    return np.mean(rs)

print("=== Retrieval recall (HUIM) ===")
for k in [10,20,30, 50, 100, 200, 300]:
    print(f"  Recall@{k}: {recall_at_k(hybrid_huim_output, ground_truth, k):.4f}")

=== Retrieval recall (HUIM) ===
  Recall@10: 0.2595
  Recall@20: 0.3194
  Recall@30: 0.3589
  Recall@50: 0.4125
  Recall@100: 0.4756
  Recall@200: 0.5550
  Recall@300: 0.6395


## 10. Save

In [11]:
pickle.dump(hybrid_huim_output, open('../model/hybrid_huim_candidates.pkl', 'wb'))
print(f"Saved: hybrid_huim_candidates.pkl  ({len(hybrid_huim_output)} users)")

Saved: hybrid_huim_candidates.pkl  (2782 users)
